# Data Preprocessing for Customer Segmentation

## Objective
This notebook prepares the raw customer data for clustering by transforming it into a clean set of numerical features suitable for K-Means. The goal is to build a reliable preprocessing pipeline that supports downstream segmentation analysis.

## 1. Import Required Libraries
The following libraries and project utilities will be used to clean, scale, and prepare the dataset for clustering.

In [7]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_loader import load_customer_data
from src.preprocessing import (
    select_clustering_features,
    impute_missing_values,
    scale_features
)

print("Data preprocessing module loaded successfully.")

Data preprocessing module loaded successfully.


## 2. Load Raw Dataset
The raw customer dataset will be loaded using the project's data loader. The original dataset should be preserved before any preprocessing steps are applied.

In [8]:
# Load the raw dataset and preserve the original data
df = load_customer_data()

df.head()
df.shape


(1000, 5)

### Dataset Overview

The raw dataset contains customer demographic and behavioral information.

Before clustering, the data must be inspected and transformed into a suitable format for machine learning.

## 3. Data Cleaning Preparation
This section outlines the preprocessing steps required to make the data suitable for clustering.

### 3.1 Feature Selection
Based on the exploratory analysis, the following variables will be removed:
- CustomerID as it is only an identifier
- Age because it is highly correlated with Annual Income
- Gender because the current clustering objective focuses on behavioral segmentation

The selected features for clustering are:
- Annual Income (k$)
- Spending Score (1-100)

In [9]:
X_selected = select_clustering_features(df)

X_selected.head()

,Annual Income (k$),Spending Score (1-100)
0,59.9,58.0
1,48.4,37.0
2,70.5,26.0
3,81.1,30.0
4,42.1,58.0


### 3.1 Handle Missing Values
Missing values are expected in numerical features. Median imputation will be used because it is more robust to outliers than mean imputation.

In [10]:
X_selected.isnull().sum()

Annual Income (k$)        4
Spending Score (1-100)    6
dtype: int64

In [11]:
X_clean = impute_missing_values(X_selected)

display(X_clean.head())
display(X_clean.isnull().sum())
display(X_clean.shape)

,Annual Income (k$),Spending Score (1-100)
0,59.9,58.0
1,48.4,37.0
2,70.5,26.0
3,81.1,30.0
4,42.1,58.0


Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64

(1000, 2)

## 4. Feature Scaling
K-Means relies on distance-based calculations, so feature scaling is necessary. The selected features have different scales, and StandardScaler will normalize their contribution to the clustering algorithm.

In [12]:
X_scaled, scaler = scale_features(X_clean)

X_scaled[:5]

array([[ 0.09745644,  0.76510194],
       [-0.3051838 , -0.28311966],
       [ 0.46858571, -0.83218813],
       [ 0.83971497, -0.63252687],
       [-0.52576062,  0.76510194]])

## 5. Validation Checks
This section includes basic validation steps to confirm that the preprocessing output is ready for modeling.

In [15]:
#Check final processed data

print("Missing values after preprocessing:")
print(X_clean.isnull().sum())

print("\n Final feature dataset shape:")
print(X_clean.shape)

print("\n Scaled data mean:")
print(X_scaled.mean(axis=0))

print("\n Scaled data standard deviation:")
print(X_scaled.std(axis=0))

Missing values after preprocessing:
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64

 Final feature dataset shape:
(1000, 2)

 Scaled data mean:
[-8.52651283e-17  1.13686838e-16]

 Scaled data standard deviation:
[1. 1.]


### Validation explanation

- We have preprocessed the data, filling the missing value with median because median is less sensitive to outliers
- Final validation results shows that the data shape is not changed (no data being dropped), mean approaching 0 and std is 1 (normalisation works).

## 6. Export Processed Dataset
The processed feature matrix can be saved for use in the modeling notebook.

In [ ]:
X_clean.to_csv(
    "../data/processed/customer_features.csv",
    index=False
)

## 7. Summary
This notebook performs the initial data preparation steps required for K-Means clustering. The pipeline focuses on cleaning missing values, selecting features aligned with the clustering objective, and scaling variables so that distance-based clustering can be applied effectively. The resulting processed data will be used as the input for the modeling notebook.